# Historical Behavioural Features

This notebook focuses on creating leakage-safe historical behavioural features for fraud detection.

The objective is to capture the past transaction behaviour of senders and receivers while ensuring that the current transaction and all future transactions are never used to calculate historical information.

The historical features will be created separately for sender and receiver accounts and validated carefully before being added to the final model dataset.

#### Load Processed Train, Validation and Test Data

The Train, Validation, and Test datasets were already created and processed in the previous notebook.

Because these datasets are large, we store them in Parquet format and load them here instead of repeating the complete preprocessing workflow.

This keeps the workflow modular and avoids repeating completed steps.

In [1]:
import pandas as pd
import numpy as np

In [2]:
train_path = "train_processed.parquet"
val_path = "validation_processed.parquet"
test_path = "test_processed.parquet"

df_train = pd.read_parquet(train_path)
df_val = pd.read_parquet(val_path)
df_test = pd.read_parquet(test_path)

print("Train shape      :", df_train.shape)
print("Validation shape :", df_val.shape)
print("Test shape       :", df_test.shape)

print("\nTrain columns:")
print(df_train.columns.tolist())

Train shape      : (6082007, 27)
Validation shape : (191147, 27)
Test shape       : (89466, 27)

Train columns:
['step', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'origin_balance_error', 'abs_origin_balance_error', 'amount_to_origin_ratio', 'sender_balance_depleted', 'destination_balance_depleted', 'origin_balance_zero', 'hour_of_day', 'day_of_week', 'is_weekend', 'sender_balance_change', 'receiver_balance_change', 'receiver_amount_difference', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


## Sender Historical Transaction Count

The sender's historical transaction count measures how many transactions the same sender had made before the current transaction.

Only transactions from strictly earlier time steps are included. Transactions occurring in the same step are excluded because they are not considered historical information at the time of prediction.

This feature can capture changes in transaction activity or sender velocity without using future information.

In [3]:
sender_step_counts = (
    df_train
    .groupby(["nameOrig", "step"], sort = True)
    .size()
    .reset_index(name="step_transaction_count")
)

sender_step_counts["sender_past_transaction_count"] = (
    sender_step_counts
    .groupby("nameOrig")["step_transaction_count"]
    .cumsum()
    .shift(1)
)

first_rows = sender_step_counts["nameOrig"].ne(
    sender_step_counts["nameOrig"].shift()
)

sender_step_counts.loc[first_rows, "sender_past_transaction_count"] = 0

sender_step_counts["sender_past_transaction_count"] = (
    sender_step_counts["sender_past_transaction_count"]
    .fillna(0)
    .astype("int32")
)

print("sender history table shape: ", sender_step_counts.shape)
print("\n Preview: ", sender_step_counts.head(10))

sender history table shape:  (6081960, 4)

 Preview:        nameOrig  step  step_transaction_count  sender_past_transaction_count
0  C1000000639   249                       1                              0
1  C1000001337   217                       1                              0
2  C1000001725    46                       1                              0
3  C1000002591   231                       1                              0
4  C1000003372   167                       1                              0
5  C1000003615   254                       1                              0
6  C1000004053   327                       1                              0
7  C1000004530    41                       1                              0
8  C1000005353   226                       1                              0
9  C1000005555   185                       1                              0


## Validate Sender Historical Count

Before merging the historical feature into the training data, we validate that the feature only contains activity from strictly earlier steps.

For transactions occurring at the same step, the current step's activity must not contribute to the historical count.

This validation is important because an apparently correct cumulative calculation can still introduce temporal leakage if the current time step is included.

In [4]:
print("\nMinimum historical count:")
print(sender_step_counts["sender_past_transaction_count"].min())

print("\nMaximum historical count:")
print(sender_step_counts["sender_past_transaction_count"].max())

print("\nMissing values:")
print(sender_step_counts["sender_past_transaction_count"].isna().sum())

print("\nNegative values:")
print((sender_step_counts["sender_past_transaction_count"] < 0).sum())

print("\nUnique senders:")
print(sender_step_counts["nameOrig"].nunique())

print("\nHistory rows:")
print(len(sender_step_counts))

print("\nValidation completed.")


Minimum historical count:
0

Maximum historical count:
2

Missing values:
0

Negative values:
0

Unique senders:
6073477

History rows:
6081960

Validation completed.


## Validate Historical Logic Using a Small Example

Before applying the historical feature calculation to the full dataset, we test the logic on a small manually verifiable example.

The expected behaviour is:

- First transaction of a sender → historical count = 0
- Multiple transactions in the same step → they must not count each other
- A later step → all transactions from earlier steps become historical

This test helps verify that the historical feature does not include current-step information.

In [5]:
test_history = pd.DataFrame({
    "nameOrig": [
        "A", "A", "A", "A", "A",
        "B", "B"
    ],
    "step": [
        1, 2, 2, 3, 4,
        5, 7
    ]
})

print("Input transactions:")
print(test_history)

test_counts = (
    test_history
    .groupby(["nameOrig", "step"])
    .size()
    .reset_index(name="step_transaction_count")
)

test_counts["sender_past_transaction_count"] = (
    test_counts
    .groupby("nameOrig")["step_transaction_count"]
    .cumsum()
    .groupby(test_counts["nameOrig"])
    .shift(1)
    .fillna(0)
    .astype(int)
)

print("\nHistorical feature result:")
print(test_counts)

Input transactions:
  nameOrig  step
0        A     1
1        A     2
2        A     2
3        A     3
4        A     4
5        B     5
6        B     7

Historical feature result:
  nameOrig  step  step_transaction_count  sender_past_transaction_count
0        A     1                       1                              0
1        A     2                       2                              1
2        A     3                       1                              3
3        A     4                       1                              4
4        B     5                       1                              0
5        B     7                       1                              1


In [6]:
df_train = df_train.merge(
    sender_step_counts[
        [
            "nameOrig",
            "step",
            "sender_past_transaction_count"
        ]
    ],
    on=["nameOrig", "step"],
    how="left"
)

print("Train shape after merge:", df_train.shape)

print("\nHistorical feature missing values:")
print(
    df_train["sender_past_transaction_count"]
    .isna()
    .sum()
)

print("\nFeature statistics:")
print(
    df_train["sender_past_transaction_count"]
    .describe()
)

Train shape after merge: (6082007, 28)

Historical feature missing values:
0

Feature statistics:
count    6.082007e+06
mean     1.396907e-03
std      3.740630e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.000000e+00
Name: sender_past_transaction_count, dtype: float64


In [7]:
print("===== SENDER HISTORY MERGE CHECK =====")

print("Expected rows :", 6082007)
print("Actual rows   :", len(df_train))

if len(df_train) == 6082007:
    print("Row count: PASSED")
else:
    print("Row count: FAILED")

print("\nMissing historical values:",
      df_train["sender_past_transaction_count"].isna().sum())

print("Duplicate transaction rows:",
      df_train.duplicated(
          subset=["step", "nameOrig", "nameDest", "amount"]
      ).sum())

===== SENDER HISTORY MERGE CHECK =====
Expected rows : 6082007
Actual rows   : 6082007
Row count: PASSED

Missing historical values: 0
Duplicate transaction rows: 0


## Sender Historical Total Amount

The sender's historical total amount measures the total transaction value associated with the sender during strictly earlier time steps.

Current-step and future transactions are excluded.

This feature captures the sender's historical transaction volume and provides context for understanding whether the current transaction is unusually large relative to the sender's previous activity.

In [8]:
sender_step_amount = (
    df_train
    .groupby(["nameOrig", "step"], sort=True)["amount"]
    .sum()
    .reset_index(name="step_total_amount")
)

sender_step_amount["sender_past_total_amount"] = (
    sender_step_amount
    .groupby("nameOrig")["step_total_amount"]
    .cumsum()
    .groupby(sender_step_amount["nameOrig"])
    .shift(1)
    .fillna(0)
)

print("Sender amount history table shape:",
      sender_step_amount.shape)

print("\nPreview:")
print(sender_step_amount.head(10))

print("\nMissing values:",
      sender_step_amount["sender_past_total_amount"].isna().sum())

print("\nMinimum:",
      sender_step_amount["sender_past_total_amount"].min())

print("\nMaximum:",
      sender_step_amount["sender_past_total_amount"].max())

Sender amount history table shape: (6081960, 4)

Preview:
      nameOrig  step  step_total_amount  sender_past_total_amount
0  C1000000639   249          244486.46                       0.0
1  C1000001337   217            3170.28                       0.0
2  C1000001725    46            8424.74                       0.0
3  C1000002591   231          261877.19                       0.0
4  C1000003372   167           20528.65                       0.0
5  C1000003615   254           49360.77                       0.0
6  C1000004053   327          211189.64                       0.0
7  C1000004530    41           93865.13                       0.0
8  C1000005353   226         3228390.11                       0.0
9  C1000005555   185          233109.79                       0.0

Missing values: 0

Minimum: 0.0

Maximum: 46211920.92


In [9]:
test_history = pd.DataFrame({
    "nameOrig": ["A", "A", "A", "A", "B", "B"],
    "step": [1, 2, 2, 3, 5, 7],
    "amount": [100, 200, 300, 400, 500, 700]
})

test_amounts = (
    test_history
    .groupby(["nameOrig", "step"])["amount"]
    .sum()
    .reset_index(name="step_total_amount")
)

test_amounts["sender_past_total_amount"] = (
    test_amounts
    .groupby("nameOrig")["step_total_amount"]
    .cumsum()
    .groupby(test_amounts["nameOrig"])
    .shift(1)
    .fillna(0)
)

print(test_amounts)

  nameOrig  step  step_total_amount  sender_past_total_amount
0        A     1                100                       0.0
1        A     2                500                     100.0
2        A     3                400                     600.0
3        B     5                500                       0.0
4        B     7                700                     500.0


In [10]:
df_train = df_train.merge(
    sender_step_amount[
        ["nameOrig", "step", "sender_past_total_amount"]
    ],
    on=["nameOrig", "step"],
    how="left"
)

print("===== SENDER PAST TOTAL AMOUNT MERGE =====")
print("Train shape:", df_train.shape)

print(
    "Missing values:",
    df_train["sender_past_total_amount"].isna().sum()
)

print(
    "Minimum:",
    df_train["sender_past_total_amount"].min()
)

print(
    "Maximum:",
    df_train["sender_past_total_amount"].max()
)

===== SENDER PAST TOTAL AMOUNT MERGE =====
Train shape: (6082007, 29)
Missing values: 0
Minimum: 0.0
Maximum: 46211920.92


In [11]:
df_train["sender_past_avg_amount"] = np.where(
    df_train["sender_past_transaction_count"] > 0,
    df_train["sender_past_total_amount"]
    / df_train["sender_past_transaction_count"],
    0
)

print("===== SENDER PAST AVERAGE AMOUNT =====")
print(
    "Missing values:",
    df_train["sender_past_avg_amount"].isna().sum()
)

print(
    "Infinite values:",
    np.isinf(df_train["sender_past_avg_amount"]).sum()
)

print(
    "\nMinimum:",
    df_train["sender_past_avg_amount"].min()
)

print(
    "Maximum:",
    df_train["sender_past_avg_amount"].max()
)

print("\nPreview:")
print(
    df_train[
        [
            "nameOrig",
            "step",
            "sender_past_transaction_count",
            "sender_past_total_amount",
            "sender_past_avg_amount"
        ]
    ].head(10)
)

===== SENDER PAST AVERAGE AMOUNT =====
Missing values: 0
Infinite values: 0

Minimum: 0.0
Maximum: 46211920.92

Preview:
      nameOrig  step  sender_past_transaction_count  sender_past_total_amount  \
0  C1231006815     1                              0                       0.0   
1  C1666544295     1                              0                       0.0   
2  C1305486145     1                              0                       0.0   
3   C840083671     1                              0                       0.0   
4  C2048537720     1                              0                       0.0   
5    C90045638     1                              0                       0.0   
6   C154988899     1                              0                       0.0   
7  C1912850431     1                              0                       0.0   
8  C1265012928     1                              0                       0.0   
9   C712410124     1                              0                  

In [12]:
sender_receiver_first_step = (
    df_train[
        ["nameOrig", "nameDest", "step"]
    ]
    .groupby(
        ["nameOrig", "nameDest"],
        as_index=False
    )["step"]
    .min()
    .rename(columns={"step": "first_step"})
)

print("===== SENDER-RECEIVER FIRST STEP =====")
print("Shape:", sender_receiver_first_step.shape)

print("\nMissing values:")
print(sender_receiver_first_step.isna().sum())

print("\nPreview:")
print(sender_receiver_first_step.head(10))

===== SENDER-RECEIVER FIRST STEP =====
Shape: (6082007, 3)

Missing values:
nameOrig      0
nameDest      0
first_step    0
dtype: int64

Preview:
      nameOrig     nameDest  first_step
0  C1000000639   C785826240         249
1  C1000001337   M216466820         217
2  C1000001725  M1974356374          46
3  C1000002591   C503690069         231
4  C1000003372  C1840417793         167
5  C1000003615   C996088808         254
6  C1000004053  C1128041097         327
7  C1000004530  C1643839147          41
8  C1000005353   C292963054         226
9  C1000005555  C1091802600         185


In [13]:
sender_step_new_receivers = (
    sender_receiver_first_step
    .groupby(["nameOrig", "first_step"])
    .size()
    .reset_index(name="new_receivers_at_step")
    .sort_values(["nameOrig", "first_step"])
)

sender_step_new_receivers["sender_past_unique_receivers"] = (
    sender_step_new_receivers
    .groupby("nameOrig")["new_receivers_at_step"]
    .cumsum()
    .groupby(sender_step_new_receivers["nameOrig"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== SENDER PAST UNIQUE RECEIVERS =====")
print(
    "History table shape:",
    sender_step_new_receivers.shape
)

print(
    "\nMissing values:",
    sender_step_new_receivers[
        "sender_past_unique_receivers"
    ].isna().sum()
)

print(
    "\nMinimum:",
    sender_step_new_receivers[
        "sender_past_unique_receivers"
    ].min()
)

print(
    "Maximum:",
    sender_step_new_receivers[
        "sender_past_unique_receivers"
    ].max()
)

print("\nPreview:")
print(sender_step_new_receivers.head(10))

===== SENDER PAST UNIQUE RECEIVERS =====
History table shape: (6081960, 4)

Missing values: 0

Minimum: 0
Maximum: 2

Preview:
      nameOrig  first_step  new_receivers_at_step  \
0  C1000000639         249                      1   
1  C1000001337         217                      1   
2  C1000001725          46                      1   
3  C1000002591         231                      1   
4  C1000003372         167                      1   
5  C1000003615         254                      1   
6  C1000004053         327                      1   
7  C1000004530          41                      1   
8  C1000005353         226                      1   
9  C1000005555         185                      1   

   sender_past_unique_receivers  
0                             0  
1                             0  
2                             0  
3                             0  
4                             0  
5                             0  
6                             0  
7                

In [14]:
test_history = pd.DataFrame({
    "nameOrig": ["A", "A", "A", "A", "A"],
    "nameDest": ["B", "C", "D", "C", "E"],
    "step": [1, 2, 2, 3, 4]
})

test_first = (
    test_history
    .groupby(["nameOrig", "nameDest"], as_index=False)["step"]
    .min()
    .rename(columns={"step": "first_step"})
)

test_new = (
    test_first
    .groupby(["nameOrig", "first_step"])
    .size()
    .reset_index(name="new_receivers_at_step")
    .sort_values(["nameOrig", "first_step"])
)

test_new["sender_past_unique_receivers"] = (
    test_new
    .groupby("nameOrig")["new_receivers_at_step"]
    .cumsum()
    .groupby(test_new["nameOrig"])
    .shift(1)
    .fillna(0)
    .astype(int)
)

print(test_new)

  nameOrig  first_step  new_receivers_at_step  sender_past_unique_receivers
0        A           1                      1                             0
1        A           2                      2                             1
2        A           4                      1                             3


In [15]:
sender_unique_receiver_history = (
    sender_step_new_receivers[
        ["nameOrig", "first_step", "sender_past_unique_receivers"]
    ]
    .rename(columns={"first_step": "step"})
)

df_train = df_train.merge(
    sender_unique_receiver_history,
    on=["nameOrig", "step"],
    how="left"
)

print("===== SENDER UNIQUE RECEIVERS MERGE =====")
print("Train shape:", df_train.shape)

print(
    "Missing values:",
    df_train["sender_past_unique_receivers"].isna().sum()
)

print(
    "Minimum:",
    df_train["sender_past_unique_receivers"].min()
)

print(
    "Maximum:",
    df_train["sender_past_unique_receivers"].max()
)

===== SENDER UNIQUE RECEIVERS MERGE =====
Train shape: (6082007, 31)
Missing values: 0
Minimum: 0
Maximum: 2


In [16]:
print("===== CURRENT TRAIN COLUMNS =====")
print("Total columns:", len(df_train.columns))

for i, col in enumerate(df_train.columns, start=1):
    print(i, col)

===== CURRENT TRAIN COLUMNS =====
Total columns: 31
1 step
2 amount
3 nameOrig
4 oldbalanceOrg
5 newbalanceOrig
6 nameDest
7 oldbalanceDest
8 newbalanceDest
9 isFraud
10 isFlaggedFraud
11 origin_balance_error
12 abs_origin_balance_error
13 amount_to_origin_ratio
14 sender_balance_depleted
15 destination_balance_depleted
16 origin_balance_zero
17 hour_of_day
18 day_of_week
19 is_weekend
20 sender_balance_change
21 receiver_balance_change
22 receiver_amount_difference
23 type_CASH_IN
24 type_CASH_OUT
25 type_DEBIT
26 type_PAYMENT
27 type_TRANSFER
28 sender_past_transaction_count
29 sender_past_total_amount
30 sender_past_avg_amount
31 sender_past_unique_receivers


In [17]:
receiver_step_counts = (
    df_train
    .groupby(["nameDest", "step"], sort=True)
    .size()
    .reset_index(name="step_transaction_count")
)

receiver_step_counts["receiver_past_transaction_count"] = (
    receiver_step_counts
    .groupby("nameDest")["step_transaction_count"]
    .cumsum()
    .groupby(receiver_step_counts["nameDest"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== RECEIVER PAST TRANSACTION COUNT =====")
print(
    "History table shape:",
    receiver_step_counts.shape
)

print(
    "\nMissing values:",
    receiver_step_counts[
        "receiver_past_transaction_count"
    ].isna().sum()
)

print(
    "Minimum:",
    receiver_step_counts[
        "receiver_past_transaction_count"
    ].min()
)

print(
    "Maximum:",
    receiver_step_counts[
        "receiver_past_transaction_count"
    ].max()
)

print("\nPreview:")
print(receiver_step_counts.head(10))

===== RECEIVER PAST TRANSACTION COUNT =====
History table shape: (5766177, 4)

Missing values: 0
Minimum: 0
Maximum: 112

Preview:
      nameDest  step  step_transaction_count  receiver_past_transaction_count
0  C1000004082   352                       1                                0
1  C1000004082   354                       1                                1
2  C1000004082   370                       1                                2
3  C1000004082   374                       1                                3
4  C1000004082   379                       1                                4
5  C1000004082   396                       1                                5
6  C1000004940   131                       1                                0
7  C1000004940   132                       1                                1
8  C1000004940   161                       1                                2
9  C1000004940   180                       1                                3


In [18]:
test_history = pd.DataFrame({
    "nameDest": ["X", "X", "X", "X", "Y", "Y"],
    "step": [1, 2, 2, 3, 5, 7]
})

test_counts = (
    test_history
    .groupby(["nameDest", "step"])
    .size()
    .reset_index(name="step_transaction_count")
)

test_counts["receiver_past_transaction_count"] = (
    test_counts
    .groupby("nameDest")["step_transaction_count"]
    .cumsum()
    .groupby(test_counts["nameDest"])
    .shift(1)
    .fillna(0)
    .astype(int)
)

print(test_counts)

  nameDest  step  step_transaction_count  receiver_past_transaction_count
0        X     1                       1                                0
1        X     2                       2                                1
2        X     3                       1                                3
3        Y     5                       1                                0
4        Y     7                       1                                1


In [19]:
df_train = df_train.merge(
    receiver_step_counts[
        ["nameDest", "step", "receiver_past_transaction_count"]
    ],
    on=["nameDest", "step"],
    how="left"
)

print("===== RECEIVER PAST TRANSACTION COUNT MERGE =====")
print("Train shape:", df_train.shape)

print(
    "Missing values:",
    df_train["receiver_past_transaction_count"].isna().sum()
)

print(
    "Minimum:",
    df_train["receiver_past_transaction_count"].min()
)

print(
    "Maximum:",
    df_train["receiver_past_transaction_count"].max()
)

===== RECEIVER PAST TRANSACTION COUNT MERGE =====
Train shape: (6082007, 32)
Missing values: 0
Minimum: 0
Maximum: 112


In [20]:
receiver_step_amount = (
    df_train
    .groupby(["nameDest", "step"], sort=True)["amount"]
    .sum()
    .reset_index(name="step_total_amount")
)

receiver_step_amount["receiver_past_total_amount"] = (
    receiver_step_amount
    .groupby("nameDest")["step_total_amount"]
    .cumsum()
    .groupby(receiver_step_amount["nameDest"])
    .shift(1)
    .fillna(0)
)

print("===== RECEIVER PAST TOTAL AMOUNT =====")
print(
    "History table shape:",
    receiver_step_amount.shape
)

print(
    "\nMissing values:",
    receiver_step_amount[
        "receiver_past_total_amount"
    ].isna().sum()
)

print(
    "Minimum:",
    receiver_step_amount[
        "receiver_past_total_amount"
    ].min()
)

print(
    "Maximum:",
    receiver_step_amount[
        "receiver_past_total_amount"
    ].max()
)

print("\nPreview:")
print(receiver_step_amount.head(10))

===== RECEIVER PAST TOTAL AMOUNT =====
History table shape: (5766177, 4)

Missing values: 0
Minimum: 0.0
Maximum: 357277441.88

Preview:
      nameDest  step  step_total_amount  receiver_past_total_amount
0  C1000004082   352          156985.31                        0.00
1  C1000004082   354          228252.33                   156985.31
2  C1000004082   370         1331742.99                   385237.64
3  C1000004082   374          363030.74                  1716980.63
4  C1000004082   379          156015.83                  2080011.37
5  C1000004082   396           23297.19                  2236027.20
6  C1000004940   131          269626.35                        0.00
7  C1000004940   132          368907.17                   269626.35
8  C1000004940   161          181650.64                   638533.52
9  C1000004940   180            4445.03                   820184.16


In [21]:
df_train = df_train.merge(
    receiver_step_amount[
        ["nameDest", "step", "receiver_past_total_amount"]
    ],
    on=["nameDest", "step"],
    how="left"
)

print("===== RECEIVER PAST TOTAL AMOUNT MERGE =====")
print("Train shape:", df_train.shape)

print(
    "Missing values:",
    df_train["receiver_past_total_amount"].isna().sum()
)

print(
    "Minimum:",
    df_train["receiver_past_total_amount"].min()
)

print(
    "Maximum:",
    df_train["receiver_past_total_amount"].max()
)

===== RECEIVER PAST TOTAL AMOUNT MERGE =====
Train shape: (6082007, 33)
Missing values: 0
Minimum: 0.0
Maximum: 357277441.88


In [22]:
df_train["receiver_past_avg_amount"] = np.where(
    df_train["receiver_past_transaction_count"] > 0,
    df_train["receiver_past_total_amount"]
    / df_train["receiver_past_transaction_count"],
    0
)

print("===== RECEIVER PAST AVERAGE AMOUNT =====")

print(
    "Missing values:",
    df_train["receiver_past_avg_amount"].isna().sum()
)

print(
    "Infinite values:",
    np.isinf(
        df_train["receiver_past_avg_amount"]
    ).sum()
)

print(
    "Minimum:",
    df_train["receiver_past_avg_amount"].min()
)

print(
    "Maximum:",
    df_train["receiver_past_avg_amount"].max()
)

print("\nPreview:")
print(
    df_train[
        [
            "nameDest",
            "step",
            "receiver_past_transaction_count",
            "receiver_past_total_amount",
            "receiver_past_avg_amount"
        ]
    ].head(10)
)

===== RECEIVER PAST AVERAGE AMOUNT =====
Missing values: 0
Infinite values: 0
Minimum: 0.0
Maximum: 39920570.215

Preview:
      nameDest  step  receiver_past_transaction_count  \
0  M1979787155     1                                0   
1  M2044282225     1                                0   
2   C553264065     1                                0   
3    C38997010     1                                0   
4  M1230701703     1                                0   
5   M573487274     1                                0   
6   M408069119     1                                0   
7   M633326333     1                                0   
8  M1176932104     1                                0   
9   C195600860     1                                0   

   receiver_past_total_amount  receiver_past_avg_amount  
0                         0.0                       0.0  
1                         0.0                       0.0  
2                         0.0                       0.0  
3                

In [23]:
receiver_sender_first_step = (
    df_train[
        ["nameDest", "nameOrig", "step"]
    ]
    .groupby(
        ["nameDest", "nameOrig"],
        as_index=False
    )["step"]
    .min()
    .rename(columns={"step": "first_step"})
)

print("===== RECEIVER-SENDER FIRST INTERACTION =====")
print(
    "History table shape:",
    receiver_sender_first_step.shape
)

print(
    "Missing values:",
    receiver_sender_first_step.isna().sum().sum()
)

print("\nPreview:")
print(receiver_sender_first_step.head(10))

===== RECEIVER-SENDER FIRST INTERACTION =====
History table shape: (6082007, 3)
Missing values: 0

Preview:
      nameDest     nameOrig  first_step
0  C1000004082  C1180747031         352
1  C1000004082  C1185840905         379
2  C1000004082  C1539355936         370
3  C1000004082  C1680720313         374
4  C1000004082  C1978911345         354
5  C1000004082   C984964842         396
6  C1000004940  C1014009702         212
7  C1000004940  C1075962941         306
8  C1000004940  C1191345963         355
9  C1000004940  C1456984251         131


In [24]:
receiver_step_new_senders = (
    receiver_sender_first_step
    .groupby(["nameDest", "first_step"])
    .size()
    .reset_index(name="new_senders_at_step")
    .sort_values(["nameDest", "first_step"])
)

receiver_step_new_senders["receiver_past_unique_senders"] = (
    receiver_step_new_senders
    .groupby("nameDest")["new_senders_at_step"]
    .cumsum()
    .groupby(receiver_step_new_senders["nameDest"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== RECEIVER PAST UNIQUE SENDERS =====")
print(
    "History table shape:",
    receiver_step_new_senders.shape
)

print(
    "Missing values:",
    receiver_step_new_senders[
        "receiver_past_unique_senders"
    ].isna().sum()
)

print(
    "Minimum:",
    receiver_step_new_senders[
        "receiver_past_unique_senders"
    ].min()
)

print(
    "Maximum:",
    receiver_step_new_senders[
        "receiver_past_unique_senders"
    ].max()
)

print("\nPreview:")
print(receiver_step_new_senders.head(10))

===== RECEIVER PAST UNIQUE SENDERS =====
History table shape: (5766177, 4)
Missing values: 0
Minimum: 0
Maximum: 112

Preview:
      nameDest  first_step  new_senders_at_step  receiver_past_unique_senders
0  C1000004082         352                    1                             0
1  C1000004082         354                    1                             1
2  C1000004082         370                    1                             2
3  C1000004082         374                    1                             3
4  C1000004082         379                    1                             4
5  C1000004082         396                    1                             5
6  C1000004940         131                    1                             0
7  C1000004940         132                    1                             1
8  C1000004940         161                    1                             2
9  C1000004940         180                    1                             3


In [25]:
test_history = pd.DataFrame({
    "nameDest": ["X", "X", "X", "X", "X", "Y", "Y"],
    "nameOrig": ["A", "B", "C", "A", "D", "P", "Q"],
    "step": [1, 2, 2, 3, 4, 5, 7]
})

test_first_step = (
    test_history
    .groupby(
        ["nameDest", "nameOrig"],
        as_index=False
    )["step"]
    .min()
    .rename(columns={"step": "first_step"})
)

test_new_senders = (
    test_first_step
    .groupby(["nameDest", "first_step"])
    .size()
    .reset_index(name="new_senders_at_step")
    .sort_values(["nameDest", "first_step"])
)

test_new_senders["receiver_past_unique_senders"] = (
    test_new_senders
    .groupby("nameDest")["new_senders_at_step"]
    .cumsum()
    .groupby(test_new_senders["nameDest"])
    .shift(1)
    .fillna(0)
    .astype(int)
)

print("===== LEAKAGE TEST =====")
print(test_new_senders)

===== LEAKAGE TEST =====
  nameDest  first_step  new_senders_at_step  receiver_past_unique_senders
0        X           1                    1                             0
1        X           2                    2                             1
2        X           4                    1                             3
3        Y           5                    1                             0
4        Y           7                    1                             1


In [26]:
receiver_unique_sender_history = (
    receiver_step_new_senders[
        [
            "nameDest",
            "first_step",
            "receiver_past_unique_senders"
        ]
    ]
    .rename(columns={"first_step": "step"})
)

df_train = df_train.merge(
    receiver_unique_sender_history,
    on=["nameDest", "step"],
    how="left"
)

print("===== RECEIVER PAST UNIQUE SENDERS MERGE =====")
print("Train shape:", df_train.shape)

print(
    "Missing values:",
    df_train[
        "receiver_past_unique_senders"
    ].isna().sum()
)

print(
    "Minimum:",
    df_train[
        "receiver_past_unique_senders"
    ].min()
)

print(
    "Maximum:",
    df_train[
        "receiver_past_unique_senders"
    ].max()
)

===== RECEIVER PAST UNIQUE SENDERS MERGE =====
Train shape: (6082007, 35)
Missing values: 0
Minimum: 0
Maximum: 112


In [27]:
history_train_val = pd.concat(
    [
        df_train[["nameDest", "step"]],
        df_val[["nameDest", "step"]]
    ],
    ignore_index=True
)

print("===== VALIDATION HISTORY BASE =====")
print("Shape:", history_train_val.shape)

print(
    "Step range:",
    history_train_val["step"].min(),
    "to",
    history_train_val["step"].max()
)

print(
    "Missing values:",
    history_train_val.isna().sum().sum()
)

===== VALIDATION HISTORY BASE =====
Shape: (6273154, 2)
Step range: 1 to 631
Missing values: 0


In [28]:
receiver_val_step_counts = (
    history_train_val
    .groupby(["nameDest", "step"], sort=True)
    .size()
    .reset_index(name="step_transaction_count")
)

receiver_val_step_counts["receiver_past_transaction_count"] = (
    receiver_val_step_counts
    .groupby("nameDest")["step_transaction_count"]
    .cumsum()
    .groupby(receiver_val_step_counts["nameDest"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== VALIDATION RECEIVER PAST TRANSACTION COUNT =====")
print(
    "History table shape:",
    receiver_val_step_counts.shape
)

print(
    "Missing values:",
    receiver_val_step_counts[
        "receiver_past_transaction_count"
    ].isna().sum()
)

print(
    "Minimum:",
    receiver_val_step_counts[
        "receiver_past_transaction_count"
    ].min()
)

print(
    "Maximum:",
    receiver_val_step_counts[
        "receiver_past_transaction_count"
    ].max()
)

print("\nPreview:")
print(receiver_val_step_counts.head(10))

===== VALIDATION RECEIVER PAST TRANSACTION COUNT =====
History table shape: (5956886, 4)
Missing values: 0
Minimum: 0
Maximum: 112

Preview:
      nameDest  step  step_transaction_count  receiver_past_transaction_count
0  C1000004082   352                       1                                0
1  C1000004082   354                       1                                1
2  C1000004082   370                       1                                2
3  C1000004082   374                       1                                3
4  C1000004082   379                       1                                4
5  C1000004082   396                       1                                5
6  C1000004940   131                       1                                0
7  C1000004940   132                       1                                1
8  C1000004940   161                       1                                2
9  C1000004940   180                       1                                3


In [29]:
df_val = df_val.merge(
    receiver_val_step_counts[
        [
            "nameDest",
            "step",
            "receiver_past_transaction_count"
        ]
    ],
    on=["nameDest", "step"],
    how="left"
)

print("===== VALIDATION RECEIVER PAST COUNT MERGE =====")
print("Validation shape:", df_val.shape)

print(
    "Missing values:",
    df_val[
        "receiver_past_transaction_count"
    ].isna().sum()
)

print(
    "Minimum:",
    df_val[
        "receiver_past_transaction_count"
    ].min()
)

print(
    "Maximum:",
    df_val[
        "receiver_past_transaction_count"
    ].max()
)

===== VALIDATION RECEIVER PAST COUNT MERGE =====
Validation shape: (191147, 28)
Missing values: 0
Minimum: 0
Maximum: 98


In [30]:
history_train_val_amount = pd.concat(
    [
        df_train[["nameDest", "step", "amount"]],
        df_val[["nameDest", "step", "amount"]]
    ],
    ignore_index=True
)

receiver_val_step_amount = (
    history_train_val_amount
    .groupby(["nameDest", "step"], sort=True)["amount"]
    .sum()
    .reset_index(name="step_total_amount")
)

receiver_val_step_amount["receiver_past_total_amount"] = (
    receiver_val_step_amount
    .groupby("nameDest")["step_total_amount"]
    .cumsum()
    .groupby(receiver_val_step_amount["nameDest"])
    .shift(1)
    .fillna(0)
)

print("===== VALIDATION RECEIVER PAST TOTAL AMOUNT =====")
print(
    "History table shape:",
    receiver_val_step_amount.shape
)

print(
    "Missing values:",
    receiver_val_step_amount[
        "receiver_past_total_amount"
    ].isna().sum()
)

print(
    "Minimum:",
    receiver_val_step_amount[
        "receiver_past_total_amount"
    ].min()
)

print(
    "Maximum:",
    receiver_val_step_amount[
        "receiver_past_total_amount"
    ].max()
)

print("\nPreview:")
print(receiver_val_step_amount.head(10))

===== VALIDATION RECEIVER PAST TOTAL AMOUNT =====
History table shape: (5956886, 4)
Missing values: 0
Minimum: 0.0
Maximum: 357277441.88

Preview:
      nameDest  step  step_total_amount  receiver_past_total_amount
0  C1000004082   352          156985.31                        0.00
1  C1000004082   354          228252.33                   156985.31
2  C1000004082   370         1331742.99                   385237.64
3  C1000004082   374          363030.74                  1716980.63
4  C1000004082   379          156015.83                  2080011.37
5  C1000004082   396           23297.19                  2236027.20
6  C1000004940   131          269626.35                        0.00
7  C1000004940   132          368907.17                   269626.35
8  C1000004940   161          181650.64                   638533.52
9  C1000004940   180            4445.03                   820184.16


In [31]:
df_val = df_val.merge(
    receiver_val_step_amount[
        [
            "nameDest",
            "step",
            "receiver_past_total_amount"
        ]
    ],
    on=["nameDest", "step"],
    how="left"
)

print("===== VALIDATION RECEIVER PAST TOTAL AMOUNT MERGE =====")
print("Validation shape:", df_val.shape)

print(
    "Missing values:",
    df_val[
        "receiver_past_total_amount"
    ].isna().sum()
)

print(
    "Minimum:",
    df_val[
        "receiver_past_total_amount"
    ].min()
)

print(
    "Maximum:",
    df_val[
        "receiver_past_total_amount"
    ].max()
)

===== VALIDATION RECEIVER PAST TOTAL AMOUNT MERGE =====
Validation shape: (191147, 29)
Missing values: 0
Minimum: 0.0
Maximum: 298744403.71


In [32]:
df_val["receiver_past_avg_amount"] = np.where(
    df_val["receiver_past_transaction_count"] > 0,
    df_val["receiver_past_total_amount"]
    / df_val["receiver_past_transaction_count"],
    0
)

print("===== VALIDATION RECEIVER PAST AVERAGE AMOUNT =====")

print(
    "Missing values:",
    df_val[
        "receiver_past_avg_amount"
    ].isna().sum()
)

print(
    "Infinite values:",
    np.isinf(
        df_val["receiver_past_avg_amount"]
    ).sum()
)

print(
    "Minimum:",
    df_val[
        "receiver_past_avg_amount"
    ].min()
)

print(
    "Maximum:",
    df_val[
        "receiver_past_avg_amount"
    ].max()
)

print("\nPreview:")
print(
    df_val[
        [
            "nameDest",
            "step",
            "receiver_past_transaction_count",
            "receiver_past_total_amount",
            "receiver_past_avg_amount"
        ]
    ].head(10)
)

===== VALIDATION RECEIVER PAST AVERAGE AMOUNT =====
Missing values: 0
Infinite values: 0
Minimum: 0.0
Maximum: 19916293.580666665

Preview:
      nameDest  step  receiver_past_transaction_count  \
0  C1180721612   521                                0   
1  C2076838677   521                                4   
2  C1859977645   521                                0   
3  C1995541082   521                                1   
4  C2005579238   521                                0   
5  C1433317922   521                                3   
6    C49314331   521                                0   
7   C387977807   521                                2   
8   C586669216   521                                0   
9  C1217435263   521                                5   

   receiver_past_total_amount  receiver_past_avg_amount  
0                        0.00                  0.000000  
1                   467860.98             116965.245000  
2                        0.00                  0.000000  


In [33]:
history_train_val_pairs = pd.concat(
    [
        df_train[["nameDest", "nameOrig", "step"]],
        df_val[["nameDest", "nameOrig", "step"]]
    ],
    ignore_index=True
)

receiver_val_sender_first_step = (
    history_train_val_pairs
    .groupby(
        ["nameDest", "nameOrig"],
        as_index=False
    )["step"]
    .min()
    .rename(columns={"step": "first_step"})
)

print("===== VALIDATION RECEIVER-SENDER FIRST INTERACTION =====")

print(
    "History table shape:",
    receiver_val_sender_first_step.shape
)

print(
    "Missing values:",
    receiver_val_sender_first_step.isna().sum().sum()
)

print("\nPreview:")
print(
    receiver_val_sender_first_step.head(10)
)

===== VALIDATION RECEIVER-SENDER FIRST INTERACTION =====
History table shape: (6273154, 3)
Missing values: 0

Preview:
      nameDest     nameOrig  first_step
0  C1000004082  C1180747031         352
1  C1000004082  C1185840905         379
2  C1000004082  C1539355936         370
3  C1000004082  C1680720313         374
4  C1000004082  C1978911345         354
5  C1000004082   C984964842         396
6  C1000004940  C1014009702         212
7  C1000004940  C1075962941         306
8  C1000004940  C1191345963         355
9  C1000004940  C1456984251         131


In [34]:
receiver_val_step_new_senders = (
    receiver_val_sender_first_step
    .groupby(["nameDest", "first_step"])
    .size()
    .reset_index(name="new_senders_at_step")
    .sort_values(["nameDest", "first_step"])
)

receiver_val_step_new_senders[
    "receiver_past_unique_senders"
] = (
    receiver_val_step_new_senders
    .groupby("nameDest")["new_senders_at_step"]
    .cumsum()
    .groupby(receiver_val_step_new_senders["nameDest"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== VALIDATION RECEIVER PAST UNIQUE SENDERS =====")

print(
    "History table shape:",
    receiver_val_step_new_senders.shape
)

print(
    "Missing values:",
    receiver_val_step_new_senders[
        "receiver_past_unique_senders"
    ].isna().sum()
)

print(
    "Minimum:",
    receiver_val_step_new_senders[
        "receiver_past_unique_senders"
    ].min()
)

print(
    "Maximum:",
    receiver_val_step_new_senders[
        "receiver_past_unique_senders"
    ].max()
)

print("\nPreview:")
print(
    receiver_val_step_new_senders.head(10)
)

===== VALIDATION RECEIVER PAST UNIQUE SENDERS =====
History table shape: (5956886, 4)
Missing values: 0
Minimum: 0
Maximum: 112

Preview:
      nameDest  first_step  new_senders_at_step  receiver_past_unique_senders
0  C1000004082         352                    1                             0
1  C1000004082         354                    1                             1
2  C1000004082         370                    1                             2
3  C1000004082         374                    1                             3
4  C1000004082         379                    1                             4
5  C1000004082         396                    1                             5
6  C1000004940         131                    1                             0
7  C1000004940         132                    1                             1
8  C1000004940         161                    1                             2
9  C1000004940         180                    1                             3


In [35]:
receiver_val_unique_sender_history = (
    receiver_val_step_new_senders[
        [
            "nameDest",
            "first_step",
            "receiver_past_unique_senders"
        ]
    ]
    .rename(columns={"first_step": "step"})
)

df_val = df_val.merge(
    receiver_val_unique_sender_history,
    on=["nameDest", "step"],
    how="left"
)

print("===== VALIDATION RECEIVER PAST UNIQUE SENDERS MERGE =====")
print("Validation shape:", df_val.shape)

print(
    "Missing values:",
    df_val[
        "receiver_past_unique_senders"
    ].isna().sum()
)

print(
    "Minimum:",
    df_val[
        "receiver_past_unique_senders"
    ].min()
)

print(
    "Maximum:",
    df_val[
        "receiver_past_unique_senders"
    ].max()
)

===== VALIDATION RECEIVER PAST UNIQUE SENDERS MERGE =====
Validation shape: (191147, 31)
Missing values: 0
Minimum: 0
Maximum: 98


In [36]:
import gc

del history_train_val
del history_train_val_amount
del history_train_val_pairs

del receiver_val_step_counts
del receiver_val_step_amount
del receiver_val_sender_first_step
del receiver_val_step_new_senders

del receiver_val_unique_sender_history

gc.collect()

print("===== MEMORY CLEANUP =====")
print("Temporary validation history objects removed successfully.")

===== MEMORY CLEANUP =====
Temporary validation history objects removed successfully.


In [37]:
history_train_val_test = pd.concat(
    [
        df_train[["nameDest", "step"]],
        df_val[["nameDest", "step"]],
        df_test[["nameDest", "step"]]
    ],
    ignore_index=True
)

print("===== TEST HISTORY BASE =====")
print("Shape:", history_train_val_test.shape)

print(
    "Step range:",
    history_train_val_test["step"].min(),
    "to",
    history_train_val_test["step"].max()
)

print(
    "Missing values:",
    history_train_val_test.isna().sum().sum()
)

===== TEST HISTORY BASE =====
Shape: (6362620, 2)
Step range: 1 to 743
Missing values: 0


In [38]:
receiver_test_step_counts = (
    history_train_val_test
    .groupby(["nameDest", "step"], sort=True)
    .size()
    .reset_index(name="step_transaction_count")
)

receiver_test_step_counts[
    "receiver_past_transaction_count"
] = (
    receiver_test_step_counts
    .groupby("nameDest")["step_transaction_count"]
    .cumsum()
    .groupby(receiver_test_step_counts["nameDest"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== TEST RECEIVER PAST TRANSACTION COUNT =====")

print(
    "History table shape:",
    receiver_test_step_counts.shape
)

print(
    "Missing values:",
    receiver_test_step_counts[
        "receiver_past_transaction_count"
    ].isna().sum()
)

print(
    "Minimum:",
    receiver_test_step_counts[
        "receiver_past_transaction_count"
    ].min()
)

print(
    "Maximum:",
    receiver_test_step_counts[
        "receiver_past_transaction_count"
    ].max()
)

print("\nPreview:")
print(
    receiver_test_step_counts.head(10)
)

===== TEST RECEIVER PAST TRANSACTION COUNT =====
History table shape: (6046187, 4)
Missing values: 0
Minimum: 0
Maximum: 112

Preview:
      nameDest  step  step_transaction_count  receiver_past_transaction_count
0  C1000004082   352                       1                                0
1  C1000004082   354                       1                                1
2  C1000004082   370                       1                                2
3  C1000004082   374                       1                                3
4  C1000004082   379                       1                                4
5  C1000004082   396                       1                                5
6  C1000004940   131                       1                                0
7  C1000004940   132                       1                                1
8  C1000004940   161                       1                                2
9  C1000004940   180                       1                                3


In [39]:
df_test = df_test.merge(
    receiver_test_step_counts[
        [
            "nameDest",
            "step",
            "receiver_past_transaction_count"
        ]
    ],
    on=["nameDest", "step"],
    how="left"
)

print("===== TEST RECEIVER PAST COUNT MERGE =====")
print("Test shape:", df_test.shape)

print(
    "Missing values:",
    df_test[
        "receiver_past_transaction_count"
    ].isna().sum()
)

print(
    "Minimum:",
    df_test[
        "receiver_past_transaction_count"
    ].min()
)

print(
    "Maximum:",
    df_test[
        "receiver_past_transaction_count"
    ].max()
)

===== TEST RECEIVER PAST COUNT MERGE =====
Test shape: (89466, 28)
Missing values: 0
Minimum: 0
Maximum: 101


In [40]:
receiver_test_step_amount = (
    pd.concat(
        [
            df_train[["nameDest", "step", "amount"]],
            df_val[["nameDest", "step", "amount"]],
            df_test[["nameDest", "step", "amount"]]
        ],
        ignore_index=True
    )
    .groupby(["nameDest", "step"], sort=True)["amount"]
    .sum()
    .reset_index(name="step_total_amount")
)

receiver_test_step_amount["receiver_past_total_amount"] = (
    receiver_test_step_amount
    .groupby("nameDest")["step_total_amount"]
    .cumsum()
    .groupby(receiver_test_step_amount["nameDest"])
    .shift(1)
    .fillna(0)
)

print("===== TEST RECEIVER PAST TOTAL AMOUNT =====")
print("History table shape:", receiver_test_step_amount.shape)
print("Missing values:", receiver_test_step_amount["receiver_past_total_amount"].isna().sum())
print("Minimum:", receiver_test_step_amount["receiver_past_total_amount"].min())
print("Maximum:", receiver_test_step_amount["receiver_past_total_amount"].max())

===== TEST RECEIVER PAST TOTAL AMOUNT =====
History table shape: (6046187, 4)
Missing values: 0
Minimum: 0.0
Maximum: 357277441.88


In [41]:
df_test = df_test.merge(
    receiver_test_step_amount[
        ["nameDest", "step", "receiver_past_total_amount"]
    ],
    on=["nameDest", "step"],
    how="left"
)

print("===== TEST RECEIVER PAST TOTAL AMOUNT MERGE =====")
print("Test shape:", df_test.shape)
print("Missing values:", df_test["receiver_past_total_amount"].isna().sum())
print("Minimum:", df_test["receiver_past_total_amount"].min())
print("Maximum:", df_test["receiver_past_total_amount"].max())

===== TEST RECEIVER PAST TOTAL AMOUNT MERGE =====
Test shape: (89466, 29)
Missing values: 0
Minimum: 0.0
Maximum: 298940794.41


In [42]:
df_test["receiver_past_avg_amount"] = np.where(
    df_test["receiver_past_transaction_count"] > 0,
    df_test["receiver_past_total_amount"]
    / df_test["receiver_past_transaction_count"],
    0
)

print("===== TEST RECEIVER PAST AVG AMOUNT =====")
print("Missing values:", df_test["receiver_past_avg_amount"].isna().sum())
print("Infinite values:", np.isinf(df_test["receiver_past_avg_amount"]).sum())
print("Minimum:", df_test["receiver_past_avg_amount"].min())
print("Maximum:", df_test["receiver_past_avg_amount"].max())

===== TEST RECEIVER PAST AVG AMOUNT =====
Missing values: 0
Infinite values: 0
Minimum: 0.0
Maximum: 19610005.38


In [43]:
history_train_val_test_pairs = pd.concat(
    [
        df_train[["nameDest", "nameOrig", "step"]],
        df_val[["nameDest", "nameOrig", "step"]],
        df_test[["nameDest", "nameOrig", "step"]]
    ],
    ignore_index=True
)

receiver_test_sender_first_step = (
    history_train_val_test_pairs
    .groupby(
        ["nameDest", "nameOrig"],
        as_index=False
    )["step"]
    .min()
    .rename(columns={"step": "first_step"})
)

print("===== TEST RECEIVER-SENDER FIRST OCCURRENCE =====")
print("Shape:", receiver_test_sender_first_step.shape)
print("Missing values:", receiver_test_sender_first_step.isna().sum().sum())

===== TEST RECEIVER-SENDER FIRST OCCURRENCE =====
Shape: (6362620, 3)
Missing values: 0


In [44]:
receiver_test_step_new_senders = (
    receiver_test_sender_first_step
    .groupby(["nameDest", "first_step"])
    .size()
    .reset_index(name="new_senders_at_step")
    .sort_values(["nameDest", "first_step"])
)

receiver_test_step_new_senders["receiver_past_unique_senders"] = (
    receiver_test_step_new_senders
    .groupby("nameDest")["new_senders_at_step"]
    .cumsum()
    .groupby(receiver_test_step_new_senders["nameDest"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== TEST RECEIVER PAST UNIQUE SENDERS =====")
print("History table shape:", receiver_test_step_new_senders.shape)
print(
    "Missing values:",
    receiver_test_step_new_senders["receiver_past_unique_senders"].isna().sum()
)
print(
    "Minimum:",
    receiver_test_step_new_senders["receiver_past_unique_senders"].min()
)
print(
    "Maximum:",
    receiver_test_step_new_senders["receiver_past_unique_senders"].max()
)

===== TEST RECEIVER PAST UNIQUE SENDERS =====
History table shape: (6046187, 4)
Missing values: 0
Minimum: 0
Maximum: 112


In [45]:
receiver_test_unique_sender_history = (
    receiver_test_step_new_senders[
        [
            "nameDest",
            "first_step",
            "receiver_past_unique_senders"
        ]
    ]
    .rename(columns={"first_step": "step"})
)

df_test = df_test.merge(
    receiver_test_unique_sender_history,
    on=["nameDest", "step"],
    how="left"
)

print("===== TEST RECEIVER PAST UNIQUE SENDERS MERGE =====")
print("Test shape:", df_test.shape)
print(
    "Missing values:",
    df_test["receiver_past_unique_senders"].isna().sum()
)
print(
    "Minimum:",
    df_test["receiver_past_unique_senders"].min()
)
print(
    "Maximum:",
    df_test["receiver_past_unique_senders"].max()
)

===== TEST RECEIVER PAST UNIQUE SENDERS MERGE =====
Test shape: (89466, 31)
Missing values: 0
Minimum: 0
Maximum: 101


In [46]:
import gc

del history_train_val_test
del history_train_val_test_pairs
del receiver_test_step_counts
del receiver_test_step_amount
del receiver_test_sender_first_step
del receiver_test_step_new_senders
del receiver_test_unique_sender_history

gc.collect()

print("===== MEMORY CLEANUP =====")
print("Temporary test receiver history objects removed successfully.")

===== MEMORY CLEANUP =====
Temporary test receiver history objects removed successfully.


In [47]:
history_train_val_sender = pd.concat(
    [
        df_train[["nameOrig", "step", "amount", "nameDest"]],
        df_val[["nameOrig", "step", "amount", "nameDest"]]
    ],
    ignore_index=True
)

print("===== VALIDATION SENDER HISTORY BASE =====")
print("Shape:", history_train_val_sender.shape)
print("Step range:", history_train_val_sender["step"].min(), "to", history_train_val_sender["step"].max())
print("Missing values:", history_train_val_sender.isna().sum().sum())

===== VALIDATION SENDER HISTORY BASE =====
Shape: (6273154, 4)
Step range: 1 to 631
Missing values: 0


In [48]:
sender_val_step_counts = (
    history_train_val_sender
    .groupby(["nameOrig", "step"], sort=True)
    .size()
    .reset_index(name="step_transaction_count")
)

sender_val_step_counts["sender_past_transaction_count"] = (
    sender_val_step_counts
    .groupby("nameOrig")["step_transaction_count"]
    .cumsum()
    .groupby(sender_val_step_counts["nameOrig"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== VALIDATION SENDER PAST TRANSACTION COUNT =====")
print("History table shape:", sender_val_step_counts.shape)
print(
    "Missing values:",
    sender_val_step_counts["sender_past_transaction_count"].isna().sum()
)
print(
    "Minimum:",
    sender_val_step_counts["sender_past_transaction_count"].min()
)
print(
    "Maximum:",
    sender_val_step_counts["sender_past_transaction_count"].max()
)

===== VALIDATION SENDER PAST TRANSACTION COUNT =====
History table shape: (6273107, 4)
Missing values: 0
Minimum: 0
Maximum: 2


In [49]:
df_val = df_val.merge(
    sender_val_step_counts[
        ["nameOrig", "step", "sender_past_transaction_count"]
    ],
    on=["nameOrig", "step"],
    how="left"
)

print("===== VALIDATION SENDER PAST COUNT MERGE =====")
print("Validation shape:", df_val.shape)
print(
    "Missing values:",
    df_val["sender_past_transaction_count"].isna().sum()
)
print(
    "Minimum:",
    df_val["sender_past_transaction_count"].min()
)
print(
    "Maximum:",
    df_val["sender_past_transaction_count"].max()
)

===== VALIDATION SENDER PAST COUNT MERGE =====
Validation shape: (191147, 32)
Missing values: 0
Minimum: 0
Maximum: 2


In [50]:
sender_val_step_amount = (
    history_train_val_sender
    .groupby(["nameOrig", "step"], sort=True)["amount"]
    .sum()
    .reset_index(name="step_total_amount")
)

sender_val_step_amount["sender_past_total_amount"] = (
    sender_val_step_amount
    .groupby("nameOrig")["step_total_amount"]
    .cumsum()
    .groupby(sender_val_step_amount["nameOrig"])
    .shift(1)
    .fillna(0)
)

print("===== VALIDATION SENDER PAST TOTAL AMOUNT =====")
print("History table shape:", sender_val_step_amount.shape)
print(
    "Missing values:",
    sender_val_step_amount["sender_past_total_amount"].isna().sum()
)
print(
    "Minimum:",
    sender_val_step_amount["sender_past_total_amount"].min()
)
print(
    "Maximum:",
    sender_val_step_amount["sender_past_total_amount"].max()
)

===== VALIDATION SENDER PAST TOTAL AMOUNT =====
History table shape: (6273107, 4)
Missing values: 0
Minimum: 0.0
Maximum: 46211920.92


In [51]:
df_val = df_val.merge(
    sender_val_step_amount[
        ["nameOrig", "step", "sender_past_total_amount"]
    ],
    on=["nameOrig", "step"],
    how="left"
)

print("===== VALIDATION SENDER PAST TOTAL AMOUNT MERGE =====")
print("Validation shape:", df_val.shape)
print(
    "Missing values:",
    df_val["sender_past_total_amount"].isna().sum()
)
print(
    "Minimum:",
    df_val["sender_past_total_amount"].min()
)
print(
    "Maximum:",
    df_val["sender_past_total_amount"].max()
)

===== VALIDATION SENDER PAST TOTAL AMOUNT MERGE =====
Validation shape: (191147, 33)
Missing values: 0
Minimum: 0.0
Maximum: 10000000.0


In [52]:
df_val["sender_past_avg_amount"] = np.where(
    df_val["sender_past_transaction_count"] > 0,
    df_val["sender_past_total_amount"]
    / df_val["sender_past_transaction_count"],
    0
)

print("===== VALIDATION SENDER PAST AVG AMOUNT =====")
print("Missing values:", df_val["sender_past_avg_amount"].isna().sum())
print("Infinite values:", np.isinf(df_val["sender_past_avg_amount"]).sum())
print("Minimum:", df_val["sender_past_avg_amount"].min())
print("Maximum:", df_val["sender_past_avg_amount"].max())

===== VALIDATION SENDER PAST AVG AMOUNT =====
Missing values: 0
Infinite values: 0
Minimum: 0.0
Maximum: 10000000.0


In [53]:
sender_val_receiver_first_step = (
    history_train_val_sender[
        ["nameOrig", "nameDest", "step"]
    ]
    .groupby(
        ["nameOrig", "nameDest"],
        as_index=False
    )["step"]
    .min()
    .rename(columns={"step": "first_step"})
)

print("===== VALIDATION SENDER-RECEIVER FIRST OCCURRENCE =====")
print("Shape:", sender_val_receiver_first_step.shape)
print(
    "Missing values:",
    sender_val_receiver_first_step.isna().sum().sum()
)

===== VALIDATION SENDER-RECEIVER FIRST OCCURRENCE =====
Shape: (6273154, 3)
Missing values: 0


In [54]:
sender_val_step_new_receivers = (
    sender_val_receiver_first_step
    .groupby(["nameOrig", "first_step"])
    .size()
    .reset_index(name="new_receivers_at_step")
    .sort_values(["nameOrig", "first_step"])
)

sender_val_step_new_receivers["sender_past_unique_receivers"] = (
    sender_val_step_new_receivers
    .groupby("nameOrig")["new_receivers_at_step"]
    .cumsum()
    .groupby(sender_val_step_new_receivers["nameOrig"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== VALIDATION SENDER PAST UNIQUE RECEIVERS =====")
print("History table shape:", sender_val_step_new_receivers.shape)
print(
    "Missing values:",
    sender_val_step_new_receivers[
        "sender_past_unique_receivers"
    ].isna().sum()
)
print(
    "Minimum:",
    sender_val_step_new_receivers[
        "sender_past_unique_receivers"
    ].min()
)
print(
    "Maximum:",
    sender_val_step_new_receivers[
        "sender_past_unique_receivers"
    ].max()
)

===== VALIDATION SENDER PAST UNIQUE RECEIVERS =====
History table shape: (6273107, 4)
Missing values: 0
Minimum: 0
Maximum: 2


In [55]:
sender_val_unique_receiver_history = (
    sender_val_step_new_receivers[
        [
            "nameOrig",
            "first_step",
            "sender_past_unique_receivers"
        ]
    ]
    .rename(columns={"first_step": "step"})
)

df_val = df_val.merge(
    sender_val_unique_receiver_history,
    on=["nameOrig", "step"],
    how="left"
)

print("===== VALIDATION SENDER PAST UNIQUE RECEIVERS MERGE =====")
print("Validation shape:", df_val.shape)
print(
    "Missing values:",
    df_val["sender_past_unique_receivers"].isna().sum()
)
print(
    "Minimum:",
    df_val["sender_past_unique_receivers"].min()
)
print(
    "Maximum:",
    df_val["sender_past_unique_receivers"].max()
)

===== VALIDATION SENDER PAST UNIQUE RECEIVERS MERGE =====
Validation shape: (191147, 35)
Missing values: 0
Minimum: 0
Maximum: 2


In [56]:
import gc

del history_train_val_sender
del sender_val_step_counts
del sender_val_step_amount
del sender_val_receiver_first_step
del sender_val_step_new_receivers
del sender_val_unique_receiver_history

gc.collect()

print("===== MEMORY CLEANUP =====")
print("Temporary validation sender history objects removed successfully.")

===== MEMORY CLEANUP =====
Temporary validation sender history objects removed successfully.


In [57]:
history_train_val_test_sender = pd.concat(
    [
        df_train[["nameOrig", "step", "amount", "nameDest"]],
        df_val[["nameOrig", "step", "amount", "nameDest"]],
        df_test[["nameOrig", "step", "amount", "nameDest"]]
    ],
    ignore_index=True
)

print("===== TEST SENDER HISTORY BASE =====")
print("Shape:", history_train_val_test_sender.shape)
print(
    "Step range:",
    history_train_val_test_sender["step"].min(),
    "to",
    history_train_val_test_sender["step"].max()
)
print(
    "Missing values:",
    history_train_val_test_sender.isna().sum().sum()
)

===== TEST SENDER HISTORY BASE =====
Shape: (6362620, 4)
Step range: 1 to 743
Missing values: 0


In [58]:
sender_test_step_counts = (
    history_train_val_test_sender
    .groupby(["nameOrig", "step"], sort=True)
    .size()
    .reset_index(name="step_transaction_count")
)

sender_test_step_counts["sender_past_transaction_count"] = (
    sender_test_step_counts
    .groupby("nameOrig")["step_transaction_count"]
    .cumsum()
    .groupby(sender_test_step_counts["nameOrig"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== TEST SENDER PAST TRANSACTION COUNT =====")
print("History table shape:", sender_test_step_counts.shape)
print(
    "Missing values:",
    sender_test_step_counts["sender_past_transaction_count"].isna().sum()
)
print(
    "Minimum:",
    sender_test_step_counts["sender_past_transaction_count"].min()
)
print(
    "Maximum:",
    sender_test_step_counts["sender_past_transaction_count"].max()
)

===== TEST SENDER PAST TRANSACTION COUNT =====
History table shape: (6362573, 4)
Missing values: 0
Minimum: 0
Maximum: 2


In [59]:
df_test = df_test.merge(
    sender_test_step_counts[
        ["nameOrig", "step", "sender_past_transaction_count"]
    ],
    on=["nameOrig", "step"],
    how="left"
)

print("===== TEST SENDER PAST COUNT MERGE =====")
print("Test shape:", df_test.shape)
print(
    "Missing values:",
    df_test["sender_past_transaction_count"].isna().sum()
)
print(
    "Minimum:",
    df_test["sender_past_transaction_count"].min()
)
print(
    "Maximum:",
    df_test["sender_past_transaction_count"].max()
)

===== TEST SENDER PAST COUNT MERGE =====
Test shape: (89466, 32)
Missing values: 0
Minimum: 0
Maximum: 1


In [60]:
sender_test_step_amount = (
    history_train_val_test_sender
    .groupby(["nameOrig", "step"], sort=True)["amount"]
    .sum()
    .reset_index(name="step_total_amount")
)

sender_test_step_amount["sender_past_total_amount"] = (
    sender_test_step_amount
    .groupby("nameOrig")["step_total_amount"]
    .cumsum()
    .groupby(sender_test_step_amount["nameOrig"])
    .shift(1)
    .fillna(0)
)

print("===== TEST SENDER PAST TOTAL AMOUNT =====")
print("History table shape:", sender_test_step_amount.shape)
print(
    "Missing values:",
    sender_test_step_amount["sender_past_total_amount"].isna().sum()
)
print(
    "Minimum:",
    sender_test_step_amount["sender_past_total_amount"].min()
)
print(
    "Maximum:",
    sender_test_step_amount["sender_past_total_amount"].max()
)

===== TEST SENDER PAST TOTAL AMOUNT =====
History table shape: (6362573, 4)
Missing values: 0
Minimum: 0.0
Maximum: 46211920.92


In [61]:
df_test = df_test.merge(
    sender_test_step_amount[
        ["nameOrig", "step", "sender_past_total_amount"]
    ],
    on=["nameOrig", "step"],
    how="left"
)

print("===== TEST SENDER PAST TOTAL AMOUNT MERGE =====")
print("Test shape:", df_test.shape)
print(
    "Missing values:",
    df_test["sender_past_total_amount"].isna().sum()
)
print(
    "Minimum:",
    df_test["sender_past_total_amount"].min()
)
print(
    "Maximum:",
    df_test["sender_past_total_amount"].max()
)

===== TEST SENDER PAST TOTAL AMOUNT MERGE =====
Test shape: (89466, 33)
Missing values: 0
Minimum: 0.0
Maximum: 4146963.53


In [62]:
df_test["sender_past_avg_amount"] = np.where(
    df_test["sender_past_transaction_count"] > 0,
    df_test["sender_past_total_amount"]
    / df_test["sender_past_transaction_count"],
    0
)

print("===== TEST SENDER PAST AVG AMOUNT =====")
print(
    "Missing values:",
    df_test["sender_past_avg_amount"].isna().sum()
)
print(
    "Infinite values:",
    np.isinf(df_test["sender_past_avg_amount"]).sum()
)
print(
    "Minimum:",
    df_test["sender_past_avg_amount"].min()
)
print(
    "Maximum:",
    df_test["sender_past_avg_amount"].max()
)

===== TEST SENDER PAST AVG AMOUNT =====
Missing values: 0
Infinite values: 0
Minimum: 0.0
Maximum: 4146963.53


In [63]:
sender_test_receiver_first_step = (
    history_train_val_test_sender[
        ["nameOrig", "nameDest", "step"]
    ]
    .groupby(
        ["nameOrig", "nameDest"],
        as_index=False
    )["step"]
    .min()
    .rename(columns={"step": "first_step"})
)

print("===== TEST SENDER-RECEIVER FIRST OCCURRENCE =====")
print("Shape:", sender_test_receiver_first_step.shape)
print(
    "Missing values:",
    sender_test_receiver_first_step.isna().sum().sum()
)

===== TEST SENDER-RECEIVER FIRST OCCURRENCE =====
Shape: (6362620, 3)
Missing values: 0


In [64]:
sender_test_step_new_receivers = (
    sender_test_receiver_first_step
    .groupby(["nameOrig", "first_step"])
    .size()
    .reset_index(name="new_receivers_at_step")
    .sort_values(["nameOrig", "first_step"])
)

sender_test_step_new_receivers["sender_past_unique_receivers"] = (
    sender_test_step_new_receivers
    .groupby("nameOrig")["new_receivers_at_step"]
    .cumsum()
    .groupby(sender_test_step_new_receivers["nameOrig"])
    .shift(1)
    .fillna(0)
    .astype("int32")
)

print("===== TEST SENDER PAST UNIQUE RECEIVERS =====")
print("History table shape:", sender_test_step_new_receivers.shape)
print(
    "Missing values:",
    sender_test_step_new_receivers[
        "sender_past_unique_receivers"
    ].isna().sum()
)
print(
    "Minimum:",
    sender_test_step_new_receivers[
        "sender_past_unique_receivers"
    ].min()
)
print(
    "Maximum:",
    sender_test_step_new_receivers[
        "sender_past_unique_receivers"
    ].max()
)

===== TEST SENDER PAST UNIQUE RECEIVERS =====
History table shape: (6362573, 4)
Missing values: 0
Minimum: 0
Maximum: 2


In [65]:
sender_test_unique_receiver_history = (
    sender_test_step_new_receivers[
        [
            "nameOrig",
            "first_step",
            "sender_past_unique_receivers"
        ]
    ]
    .rename(columns={"first_step": "step"})
)

df_test = df_test.merge(
    sender_test_unique_receiver_history,
    on=["nameOrig", "step"],
    how="left"
)

print("===== TEST SENDER PAST UNIQUE RECEIVERS MERGE =====")
print("Test shape:", df_test.shape)
print(
    "Missing values:",
    df_test["sender_past_unique_receivers"].isna().sum()
)
print(
    "Minimum:",
    df_test["sender_past_unique_receivers"].min()
)
print(
    "Maximum:",
    df_test["sender_past_unique_receivers"].max()
)

===== TEST SENDER PAST UNIQUE RECEIVERS MERGE =====
Test shape: (89466, 35)
Missing values: 0
Minimum: 0
Maximum: 1


In [66]:
import gc

del history_train_val_test_sender
del sender_test_step_counts
del sender_test_step_amount
del sender_test_receiver_first_step
del sender_test_step_new_receivers
del sender_test_unique_receiver_history

gc.collect()

print("===== MEMORY CLEANUP =====")
print("Temporary test sender history objects removed successfully.")

===== MEMORY CLEANUP =====
Temporary test sender history objects removed successfully.


In [67]:
print("===== FINAL HISTORICAL FEATURE VALIDATION =====")

print("\nTrain shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)

historical_features = [
    "sender_past_transaction_count",
    "sender_past_total_amount",
    "sender_past_avg_amount",
    "sender_past_unique_receivers",
    "receiver_past_transaction_count",
    "receiver_past_total_amount",
    "receiver_past_avg_amount",
    "receiver_past_unique_senders"
]

print("\n===== REQUIRED HISTORICAL FEATURES =====")

for feature in historical_features:
    print(
        feature,
        "->",
        feature in df_train.columns,
        feature in df_val.columns,
        feature in df_test.columns
    )

===== FINAL HISTORICAL FEATURE VALIDATION =====

Train shape: (6082007, 35)
Validation shape: (191147, 35)
Test shape: (89466, 35)

===== REQUIRED HISTORICAL FEATURES =====
sender_past_transaction_count -> True True True
sender_past_total_amount -> True True True
sender_past_avg_amount -> True True True
sender_past_unique_receivers -> True True True
receiver_past_transaction_count -> True True True
receiver_past_total_amount -> True True True
receiver_past_avg_amount -> True True True
receiver_past_unique_senders -> True True True


In [68]:
print("===== MISSING VALUE CHECK =====")

for name, data in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]:
    missing = data[historical_features].isna().sum().sum()
    infinite = np.isinf(
        data[historical_features].select_dtypes(include=np.number)
    ).sum().sum()

    print(f"{name} -> Missing: {missing}, Infinite: {infinite}")

    if missing == 0 and infinite == 0:
        print(f"{name}: PASSED")
    else:
        print(f"{name}: FAILED")

===== MISSING VALUE CHECK =====
Train -> Missing: 0, Infinite: 0
Train: PASSED
Validation -> Missing: 0, Infinite: 0
Validation: PASSED
Test -> Missing: 0, Infinite: 0
Test: PASSED


In [69]:
print("===== HISTORICAL FEATURE RANGE VALIDATION =====")

count_features = [
    "sender_past_transaction_count",
    "sender_past_unique_receivers",
    "receiver_past_transaction_count",
    "receiver_past_unique_senders"
]

amount_features = [
    "sender_past_total_amount",
    "sender_past_avg_amount",
    "receiver_past_total_amount",
    "receiver_past_avg_amount"
]

for name, data in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]:
    negative_counts = data[count_features].lt(0).sum().sum()
    negative_amounts = data[amount_features].lt(0).sum().sum()

    print(f"\n{name}")
    print("Negative count features :", negative_counts)
    print("Negative amount features:", negative_amounts)

    if negative_counts == 0 and negative_amounts == 0:
        print(f"{name}: PASSED")
    else:
        print(f"{name}: FAILED")

===== HISTORICAL FEATURE RANGE VALIDATION =====

Train
Negative count features : 0
Negative amount features: 0
Train: PASSED

Validation
Negative count features : 0
Negative amount features: 0
Validation: PASSED

Test
Negative count features : 0
Negative amount features: 0
Test: PASSED


In [70]:
print("===== FIRST TRANSACTION HISTORY CHECK =====")

for name, data in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]:
    first_sender_step = (
        data.groupby("nameOrig")["step"]
        .transform("min")
    )

    first_sender_rows = data["step"] == first_sender_step

    invalid_count = (
        data.loc[
            first_sender_rows,
            "sender_past_transaction_count"
        ] != 0
    ).sum()

    invalid_amount = (
        data.loc[
            first_sender_rows,
            "sender_past_total_amount"
        ] != 0
    ).sum()

    invalid_receivers = (
        data.loc[
            first_sender_rows,
            "sender_past_unique_receivers"
        ] != 0
    ).sum()

    print(f"\n{name}")
    print("First sender transactions:", first_sender_rows.sum())
    print("Non-zero past count      :", invalid_count)
    print("Non-zero past amount     :", invalid_amount)
    print("Non-zero unique receivers:", invalid_receivers)

    if invalid_count == 0 and invalid_amount == 0 and invalid_receivers == 0:
        print(f"{name}: PASSED")
    else:
        print(f"{name}: FAILED")

===== FIRST TRANSACTION HISTORY CHECK =====

Train
First sender transactions: 6073524
Non-zero past count      : 0
Non-zero past amount     : 0
Non-zero unique receivers: 0
Train: PASSED

Validation
First sender transactions: 191138
Non-zero past count      : 545
Non-zero past amount     : 545
Non-zero unique receivers: 545
Validation: FAILED

Test
First sender transactions: 89465
Non-zero past count      : 228
Non-zero past amount     : 228
Non-zero unique receivers: 228
Test: FAILED


In [71]:
print("===== GLOBAL FIRST TRANSACTION HISTORY CHECK =====")

combined_sender_steps = pd.concat(
    [
        df_train[["nameOrig", "step"]],
        df_val[["nameOrig", "step"]],
        df_test[["nameOrig", "step"]]
    ],
    ignore_index=True
)

global_first_step = (
    combined_sender_steps
    .groupby("nameOrig")["step"]
    .min()
)

for name, data in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]:
    first_step = data["nameOrig"].map(global_first_step)

    global_first_rows = data["step"] == first_step

    invalid_count = (
        data.loc[
            global_first_rows,
            "sender_past_transaction_count"
        ] != 0
    ).sum()

    invalid_amount = (
        data.loc[
            global_first_rows,
            "sender_past_total_amount"
        ] != 0
    ).sum()

    invalid_receivers = (
        data.loc[
            global_first_rows,
            "sender_past_unique_receivers"
        ] != 0
    ).sum()

    print(f"\n{name}")
    print("Global first transactions:", global_first_rows.sum())
    print("Non-zero past count      :", invalid_count)
    print("Non-zero past amount     :", invalid_amount)
    print("Non-zero unique receivers:", invalid_receivers)

    if (
        invalid_count == 0
        and invalid_amount == 0
        and invalid_receivers == 0
    ):
        print(f"{name}: PASSED")
    else:
        print(f"{name}: FAILED")

===== GLOBAL FIRST TRANSACTION HISTORY CHECK =====

Train
Global first transactions: 6073524
Non-zero past count      : 0
Non-zero past amount     : 0
Non-zero unique receivers: 0
Train: PASSED

Validation
Global first transactions: 190593
Non-zero past count      : 0
Non-zero past amount     : 0
Non-zero unique receivers: 0
Validation: PASSED

Test
Global first transactions: 89237
Non-zero past count      : 0
Non-zero past amount     : 0
Non-zero unique receivers: 0
Test: PASSED


In [72]:
print("===== FINAL CROSS-SPLIT INTEGRITY CHECK =====")

expected_shapes = {
    "Train": (6082007, 35),
    "Validation": (191147, 35),
    "Test": (89466, 35)
}

datasets = [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]

all_passed = True

for name, data in datasets:
    shape_ok = data.shape == expected_shapes[name]
    target_ok = "isFraud" in data.columns
    sender_ok = "nameOrig" in data.columns
    receiver_ok = "nameDest" in data.columns

    print(f"\n{name}")
    print("Shape:", data.shape, "->", "PASSED" if shape_ok else "FAILED")
    print("Target column:", "PASSED" if target_ok else "FAILED")
    print("Sender column:", "PASSED" if sender_ok else "FAILED")
    print("Receiver column:", "PASSED" if receiver_ok else "FAILED")

    if not (shape_ok and target_ok and sender_ok and receiver_ok):
        all_passed = False

print("\n===== FINAL RESULT =====")

if all_passed:
    print("ALL CROSS-SPLIT INTEGRITY CHECKS: PASSED")
else:
    print("CROSS-SPLIT INTEGRITY CHECKS: FAILED")

===== FINAL CROSS-SPLIT INTEGRITY CHECK =====

Train
Shape: (6082007, 35) -> PASSED
Target column: PASSED
Sender column: PASSED
Receiver column: PASSED

Validation
Shape: (191147, 35) -> PASSED
Target column: PASSED
Sender column: PASSED
Receiver column: PASSED

Test
Shape: (89466, 35) -> PASSED
Target column: PASSED
Sender column: PASSED
Receiver column: PASSED

===== FINAL RESULT =====
ALL CROSS-SPLIT INTEGRITY CHECKS: PASSED


## Conclusion

The historical behavioural feature engineering pipeline was successfully completed using a leakage-aware temporal methodology.

Historical features were calculated separately for sender and receiver accounts using only information from strictly earlier time steps. Transactions occurring in the same time step were excluded from historical calculations, ensuring that the current transaction does not contribute to its own historical behaviour.

The following sender-side historical features were created:

- `sender_past_transaction_count`
- `sender_past_total_amount`
- `sender_past_avg_amount`
- `sender_past_unique_receivers`

The following receiver-side historical features were created:

- `receiver_past_transaction_count`
- `receiver_past_total_amount`
- `receiver_past_avg_amount`
- `receiver_past_unique_senders`

The historical features were constructed sequentially across the chronological Train, Validation, and Test periods. Therefore, Validation transactions can use eligible history from the training period, while Test transactions can use eligible history from both Training and Validation periods. Future transactions and same-step activity are excluded.

Multiple validation checks were performed successfully:

- All required historical features are present in Train, Validation, and Test.
- No missing values were found in the historical features.
- No infinite values were found.
- No historical count or amount feature contains negative values.
- Same-step exclusion was manually verified using small controlled examples.
- Global first-transaction checks confirmed that accounts with no prior transaction have zero historical features.
- Final cross-split integrity checks passed for all datasets.

The PaySim dataset does not contain explicit bank identifiers. Therefore, historical unique-bank features could not be created and were not fabricated.

The final Train, Validation, and Test datasets contain 35 columns each and preserve their original chronological boundaries.

This completes the leakage-safe historical feature engineering stage and provides the foundation for the next stage: final feature preparation and machine learning model development.

In [73]:
df_train.to_parquet("train_historical_features.parquet", index=False)
df_val.to_parquet("validation_historical_features.parquet", index=False)
df_test.to_parquet("test_historical_features.parquet", index=False)

print("===== HISTORICAL FEATURE DATASETS SAVED =====")
print("1. train_historical_features.parquet")
print("2. validation_historical_features.parquet")
print("3. test_historical_features.parquet")

===== HISTORICAL FEATURE DATASETS SAVED =====
1. train_historical_features.parquet
2. validation_historical_features.parquet
3. test_historical_features.parquet
